In [16]:
import torch
import torch.nn as nn
import math
class LoRALayer(nn.Module):
    def __init__(self, in_dim, out_dim, rank=8, alpha=32.0):
        super().__init__()
        self.scaling = alpha/rank
        self.A = nn.Parameter(torch.zeros(in_dim, rank))
        self.B = nn.Parameter(torch.zeros(rank, out_dim))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        nn.init.zeros_(self.B)
        
    def forward(self, x):
        return (x@self.A@self.B)*self.scaling
    
    # Kaiming 均匀分布
    # B初始化为0，确保初始时LoRA不影响原模型
    # pyTorch 中用于标记可学习参数的类。被 nn.Parameter 包装的张量会自动被添加到模型的 parameters() 迭代器中，在训练时会被优化器识别并更新
    # model.named_parameters()返回模型所有可学习参数的迭代器，每个元素是元组 (参数名称, 参数对象)
    # 存在先创建参数再根据条件修改初始化方式
    # 保持不同 rank 配置下适配器的 “有效影响强度” 一致（只需调整 alpha 即可）。
    # 避免适配器输出过大干扰原模型的预训练特征，使训练更稳定。
    
class LoRALinear(nn.Module):
    def __init__(self, linear, rank=8, alpha=32):
        super.__init__()
        self.linear = linear
        self.lora = LoRALayer(linear.in_features, linear.out_features, rank, alpha)
        
        for parameter in self.linear.parameters():
            parameter.requires_grad_(False)
    def forward(self, x):
        return self.linear(x) + self.lora(x)

    # nn.Linear(in_features, out_features) 是定义线性层的构造函数 
    # Freeze the original weights 冻结原始线性层参数
    # 对位相加，将LoRA权重合并到基础权重中
    
    
def apply_lora_to_model(model, rank=8, alpha=32):
    for name,module in model.named_children():
        print(f"名称: {name}, 模块: {module.__class__.__name__}")
        if isinstance(module, nn.Linear):
            setattr(module, name, LoRALinear(module, rank, alpha))
        else:
            apply_lora_to_model(module, rank, alpha)
            
    '''
    model.named_children() 会迭代模型的直接子模块，返回每个子模块的名称和对应的模块对象
    对于每个子模块：

    如果是nn.Linear类型,则用LoRALinear(一个自定义的 LoRA 封装类)替换它
    使用setattr(model, name, ...)保持原有的模块名称，确保模型结构兼容性
    如果不是线性层,则递归调用apply_lora_to_model处理该子模块内部的层
    '''

In [25]:
def test_lora_layer():
    batch_size = 4
    in_dim = 128
    out_dim = 256
    rank =8
    
    lora_layer = LoRALayer(in_dim,out_dim)
    
    assert lora_layer.A.shape == (in_dim, rank), "A matrix has wrong shape"
    assert lora_layer.B.shape == (rank, out_dim), "B matrix has wrong shape"
    assert not torch.allclose(lora_layer.A, torch.zeros_like(lora_layer.A)), "A matrix not initialized"
    assert torch.allclose(lora_layer.B, torch.zeros_like(lora_layer.B)), "B matrix should be zero-initialized"
    
    x = torch.randn(batch_size, in_dim)
    output = lora_layer(x)
    assert output.shape == (batch_size, out_dim), "Output shape incorrect"
    
    # 如果条件为假，会触发 AssertionError

In [26]:
if __name__ == "__main__":
    test_lora_layer()